# 第8章 匿名轨迹关联与交通调查

本工作本是一份连续的实践，不另分学生任务版与参考版。按单元逐步运行，在参数区修改条件，记录自己的结果和解释；不要只执行整章脚本后截一张图。

**协议：** `ch08-tracking-v1`  
**必做：** 常速度卡尔曼预测；带门限匈牙利关联；分方向通行事件提取；YOLOX真实影像推理与独立标注核查  
**对象：** SinD地面位置：米；影像位置：像素；时间：秒；通行事件：次  
**样本：** 天津120秒平滑车辆轨迹，11531点77个源ID；每1/5/10帧对照；另用MOT17-04夜间街道35秒影像协议  
**划分：** 源ID不输入关联器，仅用于事后核查；无独立视频真值时不报告完整系统准确率  
**比较：** 固定门限5米、最长保留1.5秒；x=15米线、0.3米死区；同采样、同规则隔离关联影响

先解压完整资料包，在其目录内运行。安装 `python -m pip install -r chapters/python/video-requirements.txt`，再启动Jupyter。代码只读取固定真实数据；输出写入`outputs/chXX/`，不覆盖原数据。

每一步的“请解释”需用本次实际结果回答。默认参考配置可直接运行，但自动生成文件不代表学生已完成分析。


In [ ]:
from pathlib import Path
import sys, json
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'chapters/data').is_dir() and (p/'projects/data').is_dir())
sys.path.insert(0, str(ROOT/'chapters/python'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from threadpoolctl import threadpool_limits
from learning_support import metrics, metric_table, csv_file, primary, predictions_table, report, save_plot
THREADS = threadpool_limits(limits=1)
print('Data root:', ROOT)


## 1. 读取测得坐标，定义实验参数
这是SinD平滑地面轨迹，不是原始视频。阅读`associate`的常速度预测、卡尔曼更新和匈牙利分配，明确输入不包含源ID。


In [ ]:
from case_algorithms import associate,count_events
FILES=['projects/data/sind.json']
raw=json.loads((ROOT/FILES[0]).read_text(encoding='utf-8'))['rows']
TIMES=sorted({r[1] for r in raw})
by_time={t:[] for t in TIMES}
for row in raw:by_time[row[1]].append(row)
STRIDES=[1,5,10]
GATE=5.0
MAX_AGE=1.5
print('Coordinates / source IDs:',len(raw),len({r[0] for r in raw}))


## 2. 构造匿名帧并真实运行关联器
每帧随机重排点，防止顺序隐含源身份。修改GATE或MAX_AGE时须作为另一组配置记录；计数规则先保持不变。


In [ ]:
runs={}
for stride in STRIDES:
    kept=TIMES[::stride]
    rng=np.random.default_rng(42)
    ordered=[[by_time[t][i] for i in rng.permutation(len(by_time[t]))] for t in kept]
    frames=[(t/1000,np.array([[r[2],r[3]] for r in rows])) for t,rows in zip(kept,ordered)]
    for mode in ['position','kalman']:
        labels=associate(frames,mode,gate=GATE,max_age=MAX_AGE)
        audit=[]
        for rows,assigned in zip(ordered,labels):
            for row,new_id in zip(rows,assigned):
                audit.append([str(row[0]),new_id,row[1]/1000,row[2],row[3]])
        predicted=[[r[1],r[2],r[3],r[4]] for r in audit]
        reference=[[r[0],r[2],r[3],r[4]] for r in audit]
        runs[f'{stride}-{mode}']={'assignments':audit,'events':count_events(predicted),'reference_events':count_events(reference)}


## 3. 计算身份连续性与分方向计数
请解释：一致比例高为什么还可能存在大量碎片？总事件数一致为什么不代表100%计数正确？


In [ ]:
audit_rows=[]
for variant,run in runs.items():
    last_track={};last_source={};correct=total=switches=0
    for source,track,time,x,y in run['assignments']:
        if track in last_track and time-last_track[track][1]<=1.5:
            total+=1;correct+=last_track[track][0]==source
        if source in last_source and time-last_source[source][1]<=1.5 and track!=last_source[source][0]:switches+=1
        last_track[track]=(source,time);last_source[source]=(track,time)
    events=run['events']
    audit_rows.append([variant,len({r[1] for r in run['assignments']}),correct/total if total else None,switches,len(run['reference_events']),len(events),sum(r[2]=='+x' for r in events),sum(r[2]=='-x' for r in events)])
scores=pd.DataFrame(audit_rows,columns=['variant','generated_tracks','source_pair_agreement','ID_changes','source_events','new_events','positive','negative']);display(scores)


## 4. 定位并回看错误身份连接
按可核查错接对生成清单，逐条检查。这里不是独立人工视频标注。


In [ ]:
VIEW='10-kalman'
run=runs[VIEW];last={};wrong=[]
for source,track,time,x,y in run['assignments']:
    if track in last and source!=last[track][0] and time-last[track][1]<=1.5:
        wrong.append([track,last[track][0],source,time,x,y])
    last[track]=(source,time)
wrong_table=pd.DataFrame(wrong,columns=['new_track','previous_source','current_source','time_s','x_m','y_m']);display(wrong_table.head(10))
plt.figure(figsize=(9,4));plt.bar(scores.variant,scores.new_events,label='New IDs');plt.plot(scores.variant,scores.source_events,'ko-',label='Source ID reference')
plt.xticks(rotation=25);plt.ylabel('Crossing events');plt.legend();save_plot(ROOT,8,'identity_to_count')


## 5. 导出调查证据与人工核查记录表
输出包含每次运行的身份指派与事件表。请在核查表记录真实观察依据，不把算法输出重新命名为人工真值。


In [ ]:
primary(ROOT,8,FILES,{'strides':STRIDES,'gate_m':GATE,'max_age_s':MAX_AGE,'line_x_m':15,'band_m':.3,'count_gap_s':1.5}, {'runs':runs})
csv_file(ROOT/'outputs/ch08/events.csv',['variant','track_id','time_s','direction'],[[variant,*event] for variant,run in runs.items() for event in run['events']])
csv_file(ROOT/'outputs/ch08/wrong_associations.csv',wrong_table.columns,wrong_table.values)
csv_file(ROOT/'outputs/ch08/manual_audit.csv',['source_material','frame_or_time','object_reference','direction','observation','reviewer','status'],[['SinD trajectory (not video)',r[3],r[0],'to verify','to inspect','','pending'] for r in wrong[:10]])
report(ROOT,8,'分方向通行调查及身份误差分析',{'关联与事件':scores.to_string(index=False),'错接样例':wrong_table.head(10).to_string(index=False)},['哪一类关联错误影响了通行计数？','采样与门限是否改变结论？','哪些结论还必须用授权视频和独立标注验证？'])


## 6. 补齐真实影像链路：夜间步行街调查
这是独立影像协议`ch08-night-pedestrian-v1`，与SinD的米制轨迹不混合。按`VIDEO_LESSON.md`先确认非商业教学许可。首次安装video-requirements.txt并运行prepare_video_lesson.py下载固定模型。下方读取真实35秒视频，打印帧率和图像尺寸。


In [ ]:
import cv2
from video_lesson import DATA,PROTOCOL,detect,load_annotations,evaluate,export
capture=cv2.VideoCapture(str(DATA/'mot17-04-raw.mp4'))
print('Frames / fps / width / height:',[capture.get(k) for k in [cv2.CAP_PROP_FRAME_COUNT,cv2.CAP_PROP_FPS,cv2.CAP_PROP_FRAME_WIDTH,cv2.CAP_PROP_FRAME_HEIGHT]])
ok,frame=capture.read();capture.release();assert ok
plt.figure(figsize=(12,7));plt.imshow(cv2.cvtColor(frame,cv2.COLOR_BGR2RGB));plt.axhline(300,color='orange');plt.title('Raw night street; image counting line y=300');plt.axis('off');save_plot(ROOT,8,'real_video_first_frame')


## 7. 对真实帧运行YOLOX
模型不读取人工框或源ID，不在本片段训练。实际CPU推理需要几分钟。可阅读detect内的预处理与后处理；不要用已有结果文件冒充本次推理。


In [ ]:
from prepare_video_lesson import main as prepare_video_assets
if not (ROOT/'outputs/video-models/yolox.onnx').exists():prepare_video_assets()
detected=detect(ROOT/'outputs/video-models/yolox.onnx')
print('Actual inferred frames:',len(detected['frames']))


## 8. 独立标注核验、关联与事件评价
此时读取人工框与ID，仅用于评价。阅读evaluate中的位置归一化、卡尔曼关联、IoU匹配和事件身份匹配。两个固定阈值都报告，不能只留下更好的一组。


In [ ]:
annotations=load_annotations()
video_result=evaluate(detected,annotations)
video_scores=pd.DataFrame([{'threshold':threshold,**r['metrics']} for threshold,r in video_result['runs'].items()])
display(video_scores)
display(pd.DataFrame([[threshold,*r] for threshold,run in video_result['runs'].items() for r in run['directions']],columns=['threshold','image_direction','reference_events','model_events','bias']))


## 9. 导出并逐条回看，不将总数接近当准确
results.json独立于SinD主JSON。review.csv是待核查清单，不是已完成的人工记录。请回看至少一处漏检、错接或事件不匹配，再填写交通调查解释。


In [ ]:
export(video_result,ROOT/'outputs/ch08-video')
for threshold,run in video_result['runs'].items():
    missed=[video_result['reference_events'][i] for i in run['missed_reference']]
    print('Threshold / missed annotation-derived events:',threshold,missed[:5])
